In [1]:
!pip install --upgrade pip

In [2]:
!pip install -U torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [3]:
!pip install torch   

In [4]:
!pip install transformers

In [5]:
!pip install peft datasets accelerate bitsandbytes scikit-learn TRL huggingface_hub ijson 

In [10]:
import os
import json
import copy
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForImageTextToText,  #  최신 규격 클래스로 변경
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

# ==========================================
# 0. 사용자 설정 하이퍼파라미터
# ==========================================
print("=== [Step 0] 사용자 설정 하이퍼파라미터 ===")

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"  
DATA_FILE_PATH = "./dataset1_all.json"       
HF_REPO_ID = "parksay/qwen3-vl-ft"  
HF_TOKEN = "123"           
login(token=HF_TOKEN)



=== [Step 0] 사용자 설정 하이퍼파라미터 ===


In [12]:
# ==========================================
# 1. 데이터 로드 및 정밀 분리 (Split)
# ==========================================
print("=== [Step 1] 데이터 로드 및 10k / 2k / 1k 정밀 분리 시작 ===")


# '_' 앞뒤로 공백 추가하여 토큰 분리 유도
def apply_token_spacing(data_list):
    modified_count = 0
    for item in data_list:
        if "messages" in item:
            for message in item["messages"]:
                # user의 질문은 건드리지 않고, assistant의 정답 데이터만 치환
                if message.get("role") == "assistant" and "content" in message:
                    message["content"] = message["content"].replace("_", " _ ")
                    modified_count += 1
    print(f"✅ 전처리 완료: 총 {modified_count}건의 assistant 정답 내 '_'를 ' _ '로 치환했습니다.")
    return data_list


import ijson
import random

# 실행 시마다 항상 동일하게 무작위 분할되도록 시드 고정 (재현성 확보)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

raw_data = []
try:
    with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
        # ijson.items는 JSON 배열([]) 안의 객체({})들을 하나씩 순회하며 읽어옵니다.
        # 'item'은 배열의 각 원소를 지정하는 ijson의 고유 접두어입니다.
        parser = ijson.items(f, 'item')
        
        for idx, item in enumerate(parser):
            raw_data.append(item)
            
            # 진행 상황 출력 (10,000줄마다 완료 표시)
            if (idx + 1) % 10000 == 0:
                print(f" 현재 {idx + 1}번째 데이터 로드 완료...")

    print(f"🎉 로드 완료! 총 {len(raw_data)}개의 데이터를 성공적으로 읽었습니다.")

except ijson.common.IncompleteJSONError as e:
    print(f"\n❌ [에러] JSON 파일이 도중에 잘렸거나 닫히지 않았습니다.")
    print(f"마지막으로 읽은 데이터 인덱스: {len(raw_data)}")
    raise e
except Exception as e:
    print(f"\n❌ [에러 발생] 데이터 인덱스 약 {len(raw_data)}번째 부근을 확인하세요.")
    raise e
 
if not isinstance(raw_data, list) or "messages" not in raw_data[0]:
    raise ValueError("데이터셋 형식이 ChatML 구조의 List 형식이 아닙니다. 확인이 필요합니다.")

print(f"총 로드된 데이터 수: {len(raw_data)}건")

# 요구사항 사양 매칭 (슬라이싱 분할)
copied_data = raw_data.copy()  # 원본 데이터 보존을 위해 카피본 생성
processed_data = apply_token_spacing(copied_data)
random.shuffle(processed_data)    # 리스트 내부 원소들을 무작위로 무작위 배치

train_data = processed_data[0:10000]
val_data = processed_data[10000:12000]
test_data = processed_data[12000:13000]

print(f"-> 학습 데이터 분할 완료: {len(train_data)}건")
print(f"-> 검증 데이터 분할 완료: {len(val_data)}건")
print(f"-> 평가 데이터 분할 완료: {len(test_data)}건")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)




=== [Step 1] 데이터 로드 및 10k / 2k / 1k 정밀 분리 시작 ===
 현재 10000번째 데이터 로드 완료...
🎉 로드 완료! 총 13176개의 데이터를 성공적으로 읽었습니다.
총 로드된 데이터 수: 13176건
✅ 전처리 완료: 총 13176건의 assistant 정답 내 '_'를 ' _ '로 치환했습니다.
-> 학습 데이터 분할 완료: 10000건
-> 검증 데이터 분할 완료: 2000건
-> 평가 데이터 분할 완료: 1000건


In [13]:

# ==========================================
# 2. 토크나이저 및 노이즈 방지 설정 (좌측 패딩)
# ==========================================
print("\n=== [Step 2] 토크나이저 및 노이즈 방지 설정 ===")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"  # 배치 추론 노이즈 방지를 위한 좌측 패딩 명시

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token





=== [Step 2] 토크나이저 및 노이즈 방지 설정 ===


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [14]:
# ==========================================
# 3. Response-Only용 전처리 함수 정의 (최신 표준 규격)
# ==========================================
print("\n=== [Step 3] Response-Only 전처리 및 레이블 마스킹 (공식 Template 적용) ===")

def preprocess_chatml(example):
    messages = example["messages"]
    
    # 템플릿 깨짐을 방지하기 위해 내장 chat_template 구조 활용
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    
    tokenized_full = tokenizer(full_text, add_special_tokens=False)
    tokenized_prompt = tokenizer(prompt_text, add_special_tokens=False)
    
    input_ids = tokenized_full["input_ids"]
    prompt_len = len(tokenized_prompt["input_ids"])
    
    # 지시어 구간은 -100으로 채워 손실 계산에서 격리, 정답 구간만 유지
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    
    return {"input_ids": input_ids, "labels": labels}

# 데이터셋 최신 맵 함수 적용
tokenized_train = train_dataset.map(preprocess_chatml, remove_columns=["messages"])
tokenized_val = val_dataset.map(preprocess_chatml, remove_columns=["messages"])



=== [Step 3] Response-Only 전처리 및 레이블 마스킹 (공식 Template 적용) ===


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [15]:


# ==========================================
# 4. 좌측 패딩 맞춤형 커스텀 데이터 콜레이터 (Data Collator)
# ==========================================
print("\n=== [Step 4] QLoRA 기반 모델 양자화 및 어댑터 설정 ===")

class LeftPaddingDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        
        batch_input_ids = []
        batch_labels = []
        batch_attention_mask = []
        
        for f in features:
            ids = f["input_ids"]
            labs = f["labels"]
            pad_len = max_len - len(ids)
            
            # 요구사항: 좌측(Left) 패딩 및 마스킹 동기화 처리 유지
            padded_ids = [self.tokenizer.pad_token_id] * pad_len + ids
            padded_labs = [-100] * pad_len + labs
            attention_mask = [0] * pad_len + [1] * len(ids)
            
            batch_input_ids.append(padded_ids)
            batch_labels.append(padded_labs)
            batch_attention_mask.append(attention_mask)
            
        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long)
        }





=== [Step 4] QLoRA 기반 모델 양자화 및 어댑터 설정 ===


In [16]:
# ==========================================
# 5. QLoRA 가중치 동결 및 모델 로드
# ==========================================
print("\n=== [Step 5] Trainer 학습 파라미터 및 조기종료 설정 ===")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 캐시 비활성화 (Gradient Checkpoint 충돌 방지 최신 필수 조치)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 어댑터 레이어 설계
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()





=== [Step 5] Trainer 학습 파라미터 및 조기종료 설정 ===


model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


In [ ]:
# ==========================================
# 6. 최신 규격 파라미터(eval_strategy 적용)
# ==========================================
print("\n=== [Step 6] 독립 평가 데이터(1,000건) 기반 4대 지표 산출 ===")

training_args = TrainingArguments(
    output_dir="./qwen3_finetuned_checkpoint",
    per_device_train_batch_size=4,       
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       
    eval_strategy="steps",               # 최신 버전 인수 명명 규격으로 수정 완료 (오류 방지)
    eval_steps=200,                      
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,                   
    num_train_epochs=3,                  
    bf16=True,                           
    load_best_model_at_end=True,         
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=2,                  
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=LeftPaddingDataCollator(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

print("\n>>> 최신 최적화 파인튜닝 학습 연산을 시작합니다.")
trainer.train()
print(">>> 학습이 성공적으로 완료되었으며 최적의 가중치가 복원되었습니다.")





=== [Step 6] 독립 평가 데이터(1,000건) 기반 4대 지표 산출 ===

>>> 최신 최적화 파인튜닝 학습 연산을 시작합니다.


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,1.041377,0.951068
400,0.698916,0.658328
600,0.509681,0.518870
800,0.344466,0.457021
1000,0.284323,0.401386
1200,0.317638,0.369971


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

In [ ]:
# ==========================================
# 7. 성능 평가 지표 산출 (Accuracy, Precision, Recall, F1)
# ==========================================
print("\n=== [Step 7] 허깅페이스 허브(Hugging Face Hub) 업로드 ===")

def evaluate_performance(model, tokenizer, test_dataset):
    model.eval()
    predictions = []
    references = []
    
    print("1,000건의 평가 데이터셋에 대해 안전 추론을 진행합니다...")
    
    for idx, item in enumerate(test_dataset):
        messages = item["messages"]
        user_input = messages[0]["content"]
        true_output = messages[1]["content"].strip()
        
        # 평가 환경에서도 공식 chat_template 형식을 빌드하여 추론 노이즈 차단
        eval_messages = [{"role": "user", "content": user_input}]
        prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=30,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False  
            )
        
        input_len = inputs["input_ids"].shape[1]
        generated_tokens = output_ids[0][input_len:]
        pred_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        predictions.append(pred_output)
        references.append(true_output)
        
        if (idx + 1) % 200 == 0:
            print(f" 진행도: {idx + 1}/1000 완료...")

    # 지표 스코어 계산
    accuracy = accuracy_score(references, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        references, predictions, average='macro', zero_division=0
    )
    
    print("\n" + "="*40)
    print("        최종 모델 검증 평가서 (Test Set)")
    print("="*40)
    print(f" 1. Accuracy (정확도) : {accuracy * 100:.2f}%")
    print(f" 2. Precision (정밀도): {precision * 100:.2f}%")
    print(f" 3. Recall (재현율)   : {recall * 100:.2f}%")
    print(f" 4. F1 Score (조화평균): {f1 * 100:.2f}%")
    print("="*40 + "\n")
    
    print("--- [추론 예측 샘플 상위 3건] ---")
    for i in range(3):
        print(f"실제 정답: {references[i]} | 모델 예측: {predictions[i]}")

# 평가 구동
evaluate_performance(model, tokenizer, test_dataset)



In [ ]:

# ==========================================
# 8. 허깅페이스 업로드 및 백업
# ==========================================
if HF_REPO_ID != "parksay/qwen3-ft":
    try:
        print(f"원격 레포지토리({HF_REPO_ID})로 최적 가중치 업로드를 시작합니다...")
        model.push_to_hub(HF_REPO_ID)
        tokenizer.push_to_hub(HF_REPO_ID)
        print("✔ 허깅페이스 업로드가 최신 라이브러리 규격으로 안전하게 완료되었습니다.")
    except Exception as e:
        print(f"❌ 허깅페이스 업로드 중 오류가 발생했습니다: {e}")
else:
    print("⚠️ 허깅페이스 Repo ID가 기본값입니다. 로컬에 체크포인트를 보존합니다.")

print("\n 모든 파인튜닝 파이프라인 프로세스가 완벽하게 종료되었습니다.")